# v9c CrossJEPA — Colab training notebook

Trains **Method 1** (3D volume → 2D slice with frozen v8 ConvNeXt-Tiny teacher).

**Requirements**: Colab Pro+ with A100 (40GB) or H100. T4 is too small for the full 3D ViT at the default `(144, 192, 192)` volume — drop to `(96, 128, 128)` if you must.

Pipeline:
1. Mount Drive (checkpoints land here so a Colab disconnect doesn't lose progress).
2. Install deps (`nibabel`, `huggingface_hub`, `segmentation-models-pytorch`).
3. Unzip the source bundle.
4. Download BraTS NIfTI volumes (via HF datasets repo — adjust to your access).
5. Download the frozen v8 UNet checkpoint from HF Models.
6. Run the trainer.

Method 2 (modality → modality) requires 4 per-modality I-JEPA teachers pretrained first; see [proposals/v9c_crossjepa_IMPLEMENTATION_STATUS.md](https://github.com/archisman-das/Neuro-Lens-AI) for the runbook.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi | head -10

## 2. Install dependencies

In [ ]:
%pip install -q nibabel==5.* huggingface_hub==1.* segmentation-models-pytorch==0.* timm

## 3. Unzip the v9c CrossJEPA source bundle

In [ ]:
# Upload colab_bundle/v9c_crossjepa_bundle.zip first via the Files panel,
# OR pull it from the GitHub release.
import os, sys
BUNDLE = '/content/v9c_crossjepa_bundle.zip'
DEST = '/content/neurolens'
!rm -rf {DEST}
!mkdir -p {DEST}
!unzip -o {BUNDLE} -d {DEST}
sys.path.insert(0, DEST)
os.chdir(DEST)
!ls src/research/v9c_crossjepa/

## 4. Pull BraTS volumes from HuggingFace

BraTS 2020/2021 is gated; you need to accept the dataset agreement on HF before this works. Replace `BRATS_REPO` with a dataset you have access to.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_...'   # your token here
from huggingface_hub import snapshot_download
BRATS_REPO = 'TODO/brats-2020-mini'   # replace with your accessible dataset
BRATS_LOCAL = snapshot_download(
    repo_id=BRATS_REPO, repo_type='dataset',
    local_dir='/content/brats', allow_patterns=['*.nii.gz'],
)
!find /content/brats -name '*.nii.gz' | wc -l

## 5. Pull the frozen v8 UNet checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
V8_CKPT = hf_hub_download(
    repo_id='Tubai01/neurolens-models', repo_type='model',
    filename='attention_unet_v8/best_micro.onnx',   # will need .pt for the teacher
)
# NOTE: the teacher loader expects a PyTorch .pt; if only ONNX is
# available, rebuild the UNet from scratch with smp + load the ONNX
# weights via onnx2torch, or ship a separate .pt to the repo.
print('v8 ckpt at:', V8_CKPT)

## 6. Smoke-test the model build (no training)

Confirms the 3D ViT + predictor + (tiny mock teacher) construct and a single forward pass works on this GPU.

In [ ]:
import torch
from src.research.v9c_crossjepa.volume_to_slice import Vol2SliceModel

# Build a tiny mock teacher for the smoke test (real run uses V8FrozenTeacher.from_unet_checkpoint).
import torch.nn as nn
class TinyTeacher(nn.Module):
    embed_dim = 768
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 7, 4, 3), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 768),
        )
        for p in self.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def embed_batch(self, x): return self.encoder(x)

device = 'cuda'
model = Vol2SliceModel(
    v8_teacher=TinyTeacher().to(device),
    volume_size=(144, 192, 192), in_chans=4, patch_size=16,
).to(device)
B = 1
batch = {
    'volume': torch.randn(B, 4, 144, 192, 192, device=device),
    'target_slice_rgb': torch.rand(B, 3, 384, 384, device=device),
    'plane_idx': torch.zeros(B, dtype=torch.long, device=device),
    'slice_idx_norm': torch.rand(B, device=device),
    'voxel_spacing': torch.ones(B, 3, device=device),
    'intensity_hist': torch.rand(B, 48, device=device),
}
with torch.amp.autocast('cuda'):
    out = model.training_step(batch)
print('forward + backward smoke-test: loss=%.4f cos_sim=%.4f' % (
    float(out['loss']), float(out['cos_sim'])))
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'trainable params: {n_train/1e6:.1f} M')

## 7. Train Method 1 end-to-end

In [ ]:
!python src/train_v9c_method1_vol2slice.py --scans_glob '/content/brats/*/*.nii.gz' --v8_ckpt $V8_CKPT --output_dir /content/drive/MyDrive/v9c_crossjepa_method1 --volume_size 144 192 192 --in_channels 4 --batch_size 2 --epochs 30 --lr 2e-4 --num_workers 2 --amp --resume auto